# 8.8 只更新少量參數能改風格嗎？

# 第 8 章：風格、個性與指令遵循

前置：第7章SFT。把風格與內容分開想：想像同一位老師可以說得生動，也可以精準遵循格式。本章先用人工編寫、內容固定的短回答校準目標；LoRA是後半支線。




In [ ]:
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：原畫不重畫：加一張低rank修正圖層**

基模仍佔空間；只是微調參數變少。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/lora.svg")))

**先想一想：**r=2、in=16、out=12，adapter有幾個參數？

LoRA在凍結的Linear旁加一條低rank修正。當任務更新方向可用小矩陣表示，訓練參數可以少很多；不表示基模權重變小。

**把直覺寫成數學：**ΔW=(alpha/rank)BA；A=[r,in]，B=[out,r]，總adapter參數r(in+out)。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
from tiny_perceptron.alignment import LoRALinear

layer = LoRALinear(nn.Linear(16, 12), rank=2, alpha=2)
print("可訓練", sum(p.numel() for p in layer.parameters() if p.requires_grad))
x = torch.randn(3, 16)
assert torch.equal(layer(x), layer.base(x))
layer(x).sum().backward()
print("A/B梯度", layer.a.grad.norm().item(), layer.b.grad.norm().item())

**如何讀結果：**B初始化0，所以初始功能完全保留，A首步梯度為0是預期，不是程式壞了。

**只改一件事：**只把rank從2改為4，計算參數。
